# 🍏 Health Assistant Evaluation Demo 🍎

This notebook demonstrates how to use Azure AI Foundry's evaluation capabilities to assess the quality and safety of AI-generated health and fitness responses.

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 📊 Available Evaluators in Azure AI Foundry

Azure AI Foundry provides a comprehensive set of built-in evaluators for different aspects of AI model quality:

### **AI Quality (AI Assisted)**
- **Groundedness** - Measures how well responses are grounded in provided context
- **Relevance** - Evaluates how relevant responses are to the input query  
- **Coherence** - Assesses logical flow and consistency in responses
- **Fluency** - Measures language quality and readability
- **GPT Similarity** - Compares responses to reference answers

### **AI Quality (NLP Metrics)**
- **F1 Score** - Measures precision and recall balance
- **ROUGE Score** - Evaluates text summarization quality
- **BLEU Score** - Measures translation and generation quality
- **GLEU Score** - Google's BLEU variant for better correlation
- **METEOR Score** - Considers synonyms and stemming

### **Risk and Safety**
- **Violence** - Detects violent content
- **Sexual** - Identifies sexual content
- **Self-harm** - Detects self-harm related content
- **Hate/Unfairness** - Identifies hateful or unfair content
- **Protected Material** - Detects copyrighted content
- **Indirect Attack** - Identifies indirect prompt injection attempts

📚 **For complete details on all available evaluators, their parameters, and usage examples, visit:**  
**[Azure AI Foundry Evaluators Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/observability)**

---

# 🏋️‍♀️ Microsoft Foundry Evaluations 🏋️‍♂️

This notebook evaluates model responses locally and submits a separate cloud evaluation run to Microsoft Foundry.

## What This Notebook Does
1. **Setup and Data Creation** - Creates synthetic health and fitness Q&A data
2. **Local Evaluation** - Runs F1Score and, when configured, an AI-assisted Relevance evaluator
3. **Cloud Evaluation** - Uploads the dataset and submits a built-in F1 evaluation run

## Key Features
- **Local Evaluation** - Always runs F1Score; Relevance requires the Azure OpenAI variables
- **Cloud Evaluation** - Requires the Foundry project endpoint and Azure CLI authentication
- **Authentication** - Uses `DefaultAzureCredential`, including the active Azure CLI login
- **Error Handling** - Reports common configuration, authentication, permission, and storage issues

In [ ]:
# Setup and Data Creation
import json
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from the repository root.
root_env_path = os.environ.get("ROOT_ENV_PATH", "../../../.env")
env_loaded = load_dotenv(root_env_path)
if env_loaded:
    print(f"✅ Environment variables loaded from: {Path(root_env_path).resolve()}")
else:
    print(f"⚠️ No .env file found at: {Path(root_env_path).resolve()}")

AI_FOUNDRY_PROJECT_ENDPOINT = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME")
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
MODEL_API_VERSION = os.environ.get("MODEL_API_VERSION")

configuration = {
    "AI_FOUNDRY_PROJECT_ENDPOINT": AI_FOUNDRY_PROJECT_ENDPOINT,
    "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "MODEL_API_VERSION": MODEL_API_VERSION,
}

print("🔍 Environment Variables Status:")
for variable_name, value in configuration.items():
    print(f"   {variable_name}: {'✅ Set' if value else '❌ Missing'}")

if AI_FOUNDRY_PROJECT_ENDPOINT:
    print("\n✅ Cloud evaluation is configured.")
else:
    print("\n⚠️ Cloud evaluation will be skipped until AI_FOUNDRY_PROJECT_ENDPOINT is set.")

local_ai_variables = (
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_API_KEY,
    MODEL_DEPLOYMENT_NAME,
    MODEL_API_VERSION,
)
if all(local_ai_variables):
    print("✅ AI-assisted local Relevance evaluation is configured.")
else:
    print("ℹ️ Local evaluation will use F1Score only unless all Azure OpenAI variables are set.")

# Create synthetic health and fitness evaluation data.
synthetic_eval_data = [
    {
        "query": "How can I start a beginner workout routine at home?",
        "context": "Workout routines can include push-ups, bodyweight squats, lunges, and planks.",
        "response": "You can just go for 10 push-ups total.",
        "ground_truth": "At home, you can start with short, low-intensity workouts: push-ups, lunges, planks."
    },
    {
        "query": "Are diet sodas healthy for daily consumption?",
        "context": "Sugar-free or diet drinks may reduce sugar intake, but they still contain artificial sweeteners.",
        "response": "Yes, diet sodas are 100% healthy.",
        "ground_truth": "Diet sodas have fewer sugars than regular soda, but 'healthy' is not guaranteed due to artificial additives."
    },
    {
        "query": "What's the capital of France?",
        "context": "France is in Europe. Paris is the capital.",
        "response": "London.",
        "ground_truth": "Paris."
    }
]

# Write data to a JSONL file.
eval_data_filename = os.environ.get(
    "EVAL_DATA_FILENAME", "health_fitness_eval_data.jsonl"
)
eval_data_path = Path(eval_data_filename)
with eval_data_path.open("w", encoding="utf-8") as file:
    for row in synthetic_eval_data:
        file.write(json.dumps(row) + "\n")

print(f"✅ Evaluation data created: {eval_data_path.resolve()}")
print(f"📊 Total samples: {len(synthetic_eval_data)}")

## 🔍 Local Evaluation

Run evaluations locally using F1Score (basic text similarity) and Relevance (AI-assisted) evaluators.

In [ ]:
# Local Evaluation with Microsoft Foundry
from azure.ai.evaluation import evaluate, F1ScoreEvaluator, RelevanceEvaluator
import logging

# Reduce logging noise
logging.getLogger("promptflow").setLevel(logging.ERROR)
logging.getLogger("azure.ai.evaluation").setLevel(logging.WARNING)

print("🔍 Running Local Evaluation...")

# Configure evaluators
evaluators = {
    "f1_score": F1ScoreEvaluator()
}

evaluator_config = {
    "f1_score": {
        "column_mapping": {
            "response": "${data.response}",
            "ground_truth": "${data.ground_truth}"
        }
    }
}

# Add the AI-assisted evaluator when Azure OpenAI is configured.
# AzureOpenAIModelConfiguration expects the resource endpoint, not a deployment URL.
raw_aoai_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
aoai_base_endpoint = "/".join(raw_aoai_endpoint.split("/")[:3]) + "/" if raw_aoai_endpoint else ""
model_config = {
    "azure_endpoint": aoai_base_endpoint,
    "api_key": os.environ.get("AZURE_OPENAI_API_KEY", ""),
    "azure_deployment": os.environ.get(
        "AZURE_OPENAI_DEPLOYMENT",
        os.environ.get("MODEL_DEPLOYMENT_NAME", ""),
    ),
    "api_version": os.environ.get(
        "MODEL_API_VERSION",
        os.environ.get("AOAI_API_VERSION", os.environ.get("API_VERSION", "2025-04-01-preview")),
    ),
}

if all(
    model_config[key]
    for key in ("azure_endpoint", "api_key", "azure_deployment", "api_version")
):
    print("🤖 Adding AI-assisted Relevance evaluator...")
    evaluators["relevance"] = RelevanceEvaluator(model_config=model_config)
    evaluator_config["relevance"] = {
        "column_mapping": {
            "query": "${data.query}",
            "response": "${data.response}"
        }
    }
else:
    print("⚠️ Azure OpenAI not fully configured - using F1Score only")

# Run local evaluation
try:
    local_results_filename = os.environ.get(
        "LOCAL_RESULTS_FILENAME", "local_evaluation_results.json"
    )
    local_result = evaluate(
        data=str(eval_data_path),
        evaluators=evaluators,
        evaluator_config=evaluator_config,
        output_path=local_results_filename,
    )

    print("✅ Local evaluation completed!")
    for metric_name, value in local_result["metrics"].items():
        print(f"📊 {metric_name}: {value:.4f}")
    print(f"💾 Results saved to: {local_results_filename}")
except Exception as e:
    print(f"❌ Local evaluation failed: {e}")
    local_result = None

## ☁️ Cloud Evaluation

Upload the JSONL dataset and submit an F1 evaluation run to the Microsoft Foundry project.

In [ ]:
# Cloud Evaluation - Azure AI Projects SDK 2.x
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)
import json
import os
import time

print("☁️ Setting up Cloud Evaluation with Microsoft Foundry...")

AI_FOUNDRY_PROJECT_ENDPOINT = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
print(f"🏢 Foundry Project Endpoint: {AI_FOUNDRY_PROJECT_ENDPOINT}")

if not AI_FOUNDRY_PROJECT_ENDPOINT:
    print("⚠️ Missing AI_FOUNDRY_PROJECT_ENDPOINT in .env file")
    cloud_result = None
else:
    try:
        print("🔐 Setting up authentication...")
        credential = DefaultAzureCredential()
        project_client = AIProjectClient(
            endpoint=AI_FOUNDRY_PROJECT_ENDPOINT,
            credential=credential,
        )
        openai_client = project_client.get_openai_client()
        print("✅ Foundry clients created successfully!")

        print("📤 Uploading evaluation data to Microsoft Foundry...")
        dataset_name = os.environ.get("DATASET_NAME", "health-fitness-dataset")
        dataset_version = os.environ.get("DATASET_VERSION", str(int(time.time())))
        dataset = project_client.datasets.upload_file(
            name=dataset_name,
            version=dataset_version,
            file_path=str(eval_data_path),
        )
        print(f"✅ Data uploaded successfully! Dataset ID: {dataset.id}")

        data_source_config = DataSourceConfigCustom(
            type="custom",
            item_schema={
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "context": {"type": "string"},
                    "response": {"type": "string"},
                    "ground_truth": {"type": "string"},
                },
                "required": ["query", "response", "ground_truth"],
            },
        )

        testing_criteria = [
            TestingCriterionAzureAIEvaluator(
                type="azure_ai_evaluator",
                name="f1_score",
                evaluator_name="builtin.f1_score",
                data_mapping={
                    "response": "{{item.response}}",
                    "ground_truth": "{{item.ground_truth}}",
                },
            ),
        ]

        print("🚀 Creating and submitting cloud evaluation...")
        evaluation_name = os.environ.get(
            "EVALUATION_NAME", f"health-fitness-eval-{int(time.time())}"
        )
        eval_object = openai_client.evals.create(
            name=evaluation_name,
            data_source_config=data_source_config,
            testing_criteria=testing_criteria,
        )
        eval_run = openai_client.evals.runs.create(
            eval_id=eval_object.id,
            name=f"{evaluation_name}-run",
            data_source=CreateEvalJSONLRunDataSourceParam(
                type="jsonl",
                source=SourceFileID(type="file_id", id=dataset.id),
            ),
        )

        print("🎉 CLOUD EVALUATION SUBMITTED!")
        print(f"   📋 Evaluation ID: {eval_object.id}")
        print(f"   📋 Run ID: {eval_run.id}")
        print(f"   📋 Status: {eval_run.status}")
        print("\n🔗 View detailed results at: https://ai.azure.com/")
        print("   Navigate to your project → Evaluation → View evaluation runs")

        cloud_result = {
            "evaluation_name": evaluation_name,
            "evaluation_id": eval_object.id,
            "run_id": eval_run.id,
            "status": eval_run.status,
            "project_endpoint": AI_FOUNDRY_PROJECT_ENDPOINT,
            "dataset_id": dataset.id,
            "timestamp": int(time.time()),
        }

        cloud_results_filename = os.environ.get(
            "CLOUD_RESULTS_FILENAME", "cloud_evaluation_results.json"
        )
        with open(cloud_results_filename, "w", encoding="utf-8") as f:
            json.dump(cloud_result, f, indent=2, default=str)
        print(f"💾 Submission details saved to: {cloud_results_filename}")
        print("\n✅ SUCCESS: Cloud evaluation submitted to Microsoft Foundry!")

    except Exception as e:
        print(f"❌ Cloud evaluation failed: {e}")
        print(f"📋 Error type: {type(e).__name__}")

        error_str = str(e).lower()
        if "401" in error_str or "unauthorized" in error_str:
            print("\n🔐 AUTHENTICATION ISSUE:")
            print("   - Make sure you're logged in with: az login")
            print("   - Ensure you have access to the Foundry project")
        elif "403" in error_str or "forbidden" in error_str:
            print("\n🚫 PERMISSION ISSUE:")
            print("   - Verify you have the 'Foundry User' role")
            print("   - Check Microsoft Foundry project permissions")
        elif "404" in error_str or "not found" in error_str:
            print("\n🔍 RESOURCE NOT FOUND:")
            print("   - Verify AI_FOUNDRY_PROJECT_ENDPOINT is correct")
            print("   - Expected format: https://<account>.services.ai.azure.com/api/projects/<project>")
        elif "storage" in error_str or "blob" in error_str:
            print("\n💾 STORAGE ISSUE:")
            print("   - Ensure your Foundry project has a connected storage account")
            print("   - Check storage account permissions for the project")
        else:
            print("\n💡 TROUBLESHOOTING:")
            print(f"   - Full error: {str(e)[:300]}...")
            print("   - Confirm azure-ai-projects>=2.2.0 and openai>=2.0.0 are installed")
            print("   - Check the Microsoft Foundry project configuration")

        cloud_result = None
    finally:
        if "openai_client" in locals():
            openai_client.close()
        if "project_client" in locals():
            project_client.close()
        if "credential" in locals():
            credential.close()